# Level 1 final coding task: a three-qubit GHZ state

**Quantum Computing Foundations · Course completion · about 40 minutes**

In Module 4 you built a Bell state, which entangles two qubits. Here you extend that circuit to three qubits and make the **GHZ state**

$$|\text{GHZ}\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}$$

**How this task is graded.** Everything is graded automatically in Canvas:

1. Canvas gives you a personal parameter, θ (theta). You enter it in this notebook.
2. You build the circuit.
3. The check cell at the end tests your circuit. If every test passes, it prints a **verification value** that only a correct circuit produces for your θ. You type that value into Canvas.
4. You answer two short questions in Canvas about what your results mean.

Work through the steps in order. Run each code cell with **Shift + Enter**.

## Step 0: start Python

Like every lab in this course, this one runs entirely in your web browser: there is nothing to install and no account to create. Run the cell below with **Shift + Enter**. The first time, Python can take up to a minute to start in your browser; after that it is quick.

The labs use **qsim**, a small simulator written for this course. It uses the same names as **Qiskit**, the quantum programming toolkit used in industry and research (`QuantumCircuit`, `h`, `cx`, `measure`, `AerSimulator` and so on), so the code you write here also works in Qiskit. In Module 5 you run the same kind of code with Qiskit itself.

In [ ]:
import numpy as np
import qsim

print("Python is running in your browser. qsim version", qsim.__version__)

## Step 1: enter your parameter

Open the Canvas quiz **Final coding task: GHZ state**. Question 1 shows your θ, for example `θ = 1.25`. Type it below in place of `None`, then run the cell.

In [ ]:
THETA = None   # for example: THETA = 1.25

print("Your parameter:", THETA)

## Step 2: build the GHZ circuit

The starter code below is your Module 4 Bell circuit, written inside a function. It already has three qubits and three classical bits.

Add what is needed so that **qubit 2 joins the entangled state**, giving (|000⟩ + |111⟩)/√2 before the measurements.

Rules the check cell expects:

- exactly 3 qubits and 3 classical bits;
- measure each qubit **once, at the end**;
- the function must be called `build_ghz` and return the circuit.

In [ ]:
from qsim import QuantumCircuit

def build_ghz():
    qc = QuantumCircuit(3, 3)

    # Module 4 Bell circuit: H on qubit 0, then CNOT from qubit 0 to qubit 1
    qc.h(0)
    qc.cx(0, 1)

    # YOUR CODE: make qubit 2 part of the same entangled state


    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

build_ghz().draw()

## Step 3: run it and look at the counts

Run your circuit 1,000 times on the simulator. Before you run it, predict: which outcomes should appear, and how often?

In [ ]:
import matplotlib.pyplot as plt   # used by plot_histogram
from qsim import transpile, AerSimulator, plot_histogram

sim = AerSimulator()
counts = sim.run(transpile(build_ghz(), sim), shots=1000).result().get_counts()
print(counts)
plot_histogram(counts)

Look at your counts and think about two questions; Canvas asks about them in Step 5:

- If you knew only the result of qubit 0, what could you say about qubits 1 and 2?
- Would these counts alone prove the three qubits share one entangled state?

## Step 4: run the check cell

**Do not edit this cell.** Run it. It checks that:

1. your circuit runs and gives only `000` and `111`, each about half the time;
2. the state just before the measurements is the GHZ state (fidelity at least 0.99).

If both pass, it prints your verification value. Type it into Canvas question 1 with three decimal places, for example `0.315` or `-0.955`. If a test does not pass, read the message, fix your circuit in Step 2, run Step 2 again, then run this cell again.

In [ ]:
# CHECK CELL: do not edit
import numpy as np
from qsim import QuantumCircuit, AerSimulator, Statevector, state_fidelity, simulate_density_matrix

SHOTS = 1000
SEED = 2026
FIDELITY_MIN = 0.99
# Three standard deviations of shot noise for p = 0.5 at 1,000 shots: 3 * sqrt(0.25 / 1000) = 0.047
COUNT_TOLERANCE = 0.047


def _target_ghz():
    sv = np.zeros(8, dtype=complex)
    sv[0] = sv[7] = 1 / np.sqrt(2)
    return Statevector(sv)


def _strip_final_measurements(qc):
    """Remove only the last measurement on each qubit, if nothing but barriers follows it.

    Any other measurement or reset stays in, so a mid-circuit measurement (which turns the state
    into a classical mixture) is still simulated.
    """
    keep = list(qc.data)
    done = set()
    for idx in range(len(keep) - 1, -1, -1):
        inst = keep[idx]
        if inst.name == "barrier":
            continue
        if inst.name == "measure" and inst.qubits[0] not in done:
            keep[idx] = None
            done.add(inst.qubits[0])
            continue
        done.update(inst.qubits)
        if len(done) == qc.num_qubits:
            break
    body = qc.copy_empty_like()
    for inst in keep:
        if inst is not None:
            body.append(inst)
    return body


def check_ghz(qc, theta):
    msgs = []
    if not isinstance(qc, QuantumCircuit):
        return False, ["build_ghz() must return a QuantumCircuit."], None
    if qc.num_qubits != 3:
        return False, [f"The circuit has {qc.num_qubits} qubits; the GHZ task needs exactly 3."], None
    if not any(inst.name == "measure" for inst in qc.data):
        return False, ["Add measurements of all three qubits at the end of the circuit."], None
    if qc.num_clbits != 3:
        return False, [f"The circuit has {qc.num_clbits} classical bits; use 3. Use either QuantumCircuit(3, 3) with "
                       "qc.measure([0, 1, 2], [0, 1, 2]), or QuantumCircuit(3) with qc.measure_all(), not both."], None

    # Test 1: the circuit runs and gives the expected counts.
    try:
        counts = AerSimulator(seed_simulator=SEED).run(qc, shots=SHOTS).result().get_counts()
    except Exception as exc:  # noqa: BLE001
        return False, [f"The circuit did not run: {exc}"], None
    bad = {k: v for k, v in counts.items() if k.replace(" ", "") not in ("000", "111")}
    p000 = sum(v for k, v in counts.items() if k.replace(" ", "") == "000") / SHOTS
    counts_ok = not bad and abs(p000 - 0.5) <= COUNT_TOLERANCE
    if bad:
        msgs.append(f"Unexpected outcomes {sorted(bad)}: a GHZ state gives only 000 and 111.")
    if abs(p000 - 0.5) > COUNT_TOLERANCE:
        msgs.append(f"000 appeared with probability {p000:.3f}; a GHZ state gives about 0.5.")
    msgs.append(f"Test 1, counts: {'passed' if counts_ok else 'not passed'} ({dict(counts)})")

    # Test 2: the state before measurement is the GHZ state, up to global phase.
    rho = simulate_density_matrix(_strip_final_measurements(qc))
    fid = state_fidelity(rho, _target_ghz())
    if fid < FIDELITY_MIN:
        hint = ("Your counts look right, but equal counts of 000 and 111 are not enough: the three qubits must "
                "share one coherent state, (|000> + |111>)/sqrt(2), and each qubit is measured only once, at the end."
                if counts_ok else "The target before measurement is (|000> + |111>)/sqrt(2).")
        msgs.append(f"Test 2, state: not passed (fidelity with the GHZ state {fid:.3f}, needs at least {FIDELITY_MIN}). {hint}")
    else:
        msgs.append(f"Test 2, state: passed (fidelity {fid:.3f})")

    if not (counts_ok and fid >= FIDELITY_MIN):
        return False, msgs, None

    # Verification value: <XXX> after a phase rotation Rz(theta) on qubit 0, from the learner's own state.
    rz = QuantumCircuit(3)
    rz.rz(theta, 0)
    value = float(np.real(rho.evolve(rz).expectation_value("XXX")))
    return True, msgs, round(value, 3)


if THETA is None:
    print("Enter your parameter THETA in Step 1 first, then run this cell again.")
else:
    ok, messages, value = check_ghz(build_ghz(), float(THETA))
    for m in messages:
        print(m)
    if ok:
        print(f"\nAll tests passed. Your verification value for theta = {float(THETA)}: {value:.3f}")
        print("Type this value into Canvas question 1.")
    else:
        print("\nNot passed yet: no verification value. Fix your circuit in Step 2 and run both cells again.")


## Step 5: finish in Canvas

1. Type your verification value into question 1.
2. Answer questions 2 and 3 about your results.
3. Submit the quiz.

If you take the quiz again, Canvas may show a new θ. Enter the new value in Step 1, run the check cell again, and use the new verification value.

Your notebook is saved in this browser as you work. It is not saved anywhere else, so to keep a copy choose **File > Download**. If you open the lab in a different browser or on another computer, you start from a fresh copy.